# Divorce Analysis — Data Preparation

**Changes in Socioeconomic Characteristics Associated with Recent Divorce in the United States in 2008, 2016 & 2024**

This notebook turns the raw IPUMS USA extract (`usa_00008`) into an analysis-ready CSV:

1. **Parse** the IPUMS DDI codebook (`usa_00008.xml`) to get each variable's fixed-width position, implied decimals, and value labels.
2. **Decode** the fixed-width data file (`usa_00008.dat` or `usa_00008.dat.gz`) into `usa_00008_decoded.csv`, adding a `*_LABEL` column for every categorical variable.
3. **Clean** the decoded file by dropping technical fields and redundant code/label columns → `usa_00008_cleaned.csv`.

### Required input files

| File | Where to get it |
|---|---|
| `usa_00008.xml` | In this repository |
| `usa_00008.dat.gz` (or the unzipped `usa_00008.dat`) | IPUMS USA extract download (not committed to git — too large) |

### Running on Google Colab
Upload both files to a folder in Google Drive (default: `MyDrive/divorce-analysis/`) and run all cells. The setup cell mounts Drive automatically. Edit `COLAB_DATA_DIR` below if you use a different folder.

### Running on Kaggle
Upload both files as a Kaggle Dataset and attach it to the notebook (**Add Input**). The setup cell searches `/kaggle/input` for them automatically. Outputs are written to `/kaggle/working`.

### Running locally
Put the files next to this notebook.

## 0. Setup and environment detection

In [4]:
import gzip
import os
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
IN_KAGGLE = os.path.exists("/kaggle/input")

# Folder in Google Drive that holds usa_00008.xml and usa_00008.dat(.gz)
COLAB_DATA_DIR = "/content/drive/MyDrive/divorce-analysis"

XML_NAME = "usa_00008.xml"
DAT_NAMES = ["usa_00008.dat", "usa_00008.dat.gz"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(COLAB_DATA_DIR)
    WORK_DIR = Path("/content")
elif IN_KAGGLE:
    # Find the attached dataset that contains the codebook
    matches = list(Path("/kaggle/input").rglob(XML_NAME))
    if not matches:
        raise FileNotFoundError(f"{XML_NAME} not found under /kaggle/input — attach the dataset first.")
    DATA_DIR = matches[0].parent
    WORK_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path.cwd()
    WORK_DIR = Path.cwd()

XML_FILE = DATA_DIR / XML_NAME

# Prefer the uncompressed .dat (faster), fall back to .dat.gz
DAT_FILE = next(
    (DATA_DIR / n for n in DAT_NAMES if (DATA_DIR / n).exists()),
    None,
)

DECODED_FILE = WORK_DIR / "usa_00008_decoded.csv"
CLEANED_FILE = WORK_DIR / "usa_00008_cleaned.csv"

if not XML_FILE.exists():
    raise FileNotFoundError(f"Missing codebook: {XML_FILE}")
if DAT_FILE is None:
    raise FileNotFoundError(f"Missing data file: one of {DAT_NAMES} in {DATA_DIR}")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")
print(f"Codebook:    {XML_FILE}")
print(f"Data file:   {DAT_FILE} ({DAT_FILE.stat().st_size / 1024**2:,.0f} MB)")
print(f"Decoded out: {DECODED_FILE}")
print(f"Cleaned out: {CLEANED_FILE}")

Environment: Local
pandas 3.0.6, numpy 2.4.3
Codebook:    c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008.xml
Data file:   c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008.dat (947 MB)
Decoded out: c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008_decoded.csv
Cleaned out: c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008_cleaned.csv


## 1. Parse the IPUMS DDI codebook (XML)

Each `<var>` element gives the variable's column positions in the fixed-width file (`StartPos`/`EndPos`), the number of implied decimals (`dcml`), and — for categorical variables — the code → label mapping.

In [5]:
print("Reading XML metadata...")

tree = ET.parse(XML_FILE)
root = tree.getroot()

ns = {"ddi": "ddi:codebook:2_5"}

variables = []

for var in root.findall(".//ddi:dataDscr/ddi:var", ns):

    name = var.get("name")

    loc = var.find("ddi:location", ns)

    start = int(loc.get("StartPos")) - 1
    end = int(loc.get("EndPos"))

    decimals = int(var.get("dcml", "0"))

    labels = {}

    for cat in var.findall("ddi:catgry", ns):

        value_node = cat.find("ddi:catValu", ns)
        label_node = cat.find("ddi:labl", ns)

        if value_node is None or value_node.text is None:
            continue

        value = value_node.text.strip()

        label = (
            label_node.text.strip()
            if label_node is not None and label_node.text
            else ""
        )

        labels[value] = label

    variables.append({
        "name": name,
        "start": start,
        "end": end,
        "width": end - start,
        "decimals": decimals,
        "labels": labels
    })


print(f"Found {len(variables)} IPUMS variables.")

Reading XML metadata...
Found 46 IPUMS variables.


In [6]:
# Codebook overview: layout of every variable in the fixed-width file
codebook = pd.DataFrame([
    {
        "name": v["name"],
        "start_pos": v["start"] + 1,
        "end_pos": v["end"],
        "width": v["width"],
        "decimals": v["decimals"],
        "n_labels": len(v["labels"]),
        "type": "categorical" if v["labels"] else "numeric",
    }
    for v in variables
])
codebook

,name,start_pos,end_pos,width,decimals,n_labels,type
0,YEAR,1,4,4,0,39,categorical
1,SAMPLE,5,10,6,0,154,categorical
2,SERIAL,11,18,8,0,0,numeric
3,CBSERIAL,19,31,13,0,0,numeric
4,HHWT,32,41,10,2,0,numeric
5,HHTYPE,42,42,1,0,9,categorical
6,CLUSTER,43,55,13,0,0,numeric
7,REGION,56,57,2,0,17,categorical
8,STATEFIP,58,59,2,0,62,categorical
9,METRO,60,60,1,0,5,categorical


## 2. Decode the fixed-width DAT file → CSV

The file is ~1 GB uncompressed (several million rows), so it is processed in chunks and appended to the output CSV. Categorical codes are kept as strings (to preserve leading zeros such as `01`) and a readable `*_LABEL` column is added next to each one. Numeric variables are converted to numbers and IPUMS implied decimals are applied.

In [9]:
# Increase if you have plenty of RAM.
# 250k is reasonable for ~16 GB RAM (Colab/Kaggle default sessions are fine).
CHUNK_SIZE = 250_000


def decode_chunk(lines):

    data = {}

    for var in variables:

        name = var["name"]
        start = var["start"]
        end = var["end"]

        values = [
            line[start:end].strip()
            for line in lines
        ]

        # Preserve raw codes initially
        data[name] = values

        # Add readable labels
        if var["labels"]:

            mapping = var["labels"]

            data[name + "_LABEL"] = [
                mapping.get(v, "")
                for v in values
            ]

    df = pd.DataFrame(data)

    # -----------------------------------------------------
    # Numeric conversion
    # -----------------------------------------------------

    for var in variables:

        name = var["name"]

        # Categorical codes remain as strings.
        # Important because codes such as 01 must retain
        # their leading zeros.
        if var["labels"]:
            continue

        df[name] = pd.to_numeric(
            df[name],
            errors="coerce"
        )

        # IPUMS implied decimals
        if var["decimals"] > 0:
            df[name] /= 10 ** var["decimals"]

    return df


def open_dat(path):
    # Read either the plain .dat or the gzipped download directly
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="ISO-8859-1")
    return open(path, "r", encoding="ISO-8859-1", buffering=1024 * 1024 * 16)

In [10]:
start_time = time.time()

rows_processed = 0
first_chunk = True


def write_chunk(batch):
    global rows_processed, first_chunk

    df = decode_chunk(batch)

    df.to_csv(
        DECODED_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    rows_processed += len(batch)
    first_chunk = False


if DECODED_FILE.exists():
    DECODED_FILE.unlink()


with open_dat(DAT_FILE) as f:

    batch = []

    for line in f:

        batch.append(line.rstrip("\r\n"))

        if len(batch) == CHUNK_SIZE:

            write_chunk(batch)
            batch.clear()

            elapsed = time.time() - start_time

            print(
                f"{rows_processed:,} rows processed | "
                f"{rows_processed / elapsed:,.0f} rows/sec | "
                f"{elapsed / 60:.1f} min elapsed"
            )

    # Final partial chunk
    if batch:
        write_chunk(batch)


elapsed = time.time() - start_time

size_gb = DECODED_FILE.stat().st_size / (1024 ** 3)

print()
print("========================================")
print("Conversion complete")
print("========================================")
print(f"Rows:        {rows_processed:,}")
print(f"Variables:   {len(variables)}")
print(f"Output:      {DECODED_FILE}")
print(f"CSV size:    {size_gb:.2f} GB")
print(f"Runtime:     {elapsed / 60:.2f} minutes")
print(
    f"Average:     "
    f"{rows_processed / elapsed:,.0f} rows/sec"
)

250,000 rows processed | 30,228 rows/sec | 0.1 min elapsed
500,000 rows processed | 31,303 rows/sec | 0.3 min elapsed
750,000 rows processed | 31,777 rows/sec | 0.4 min elapsed
1,000,000 rows processed | 31,942 rows/sec | 0.5 min elapsed
1,250,000 rows processed | 32,127 rows/sec | 0.6 min elapsed
1,500,000 rows processed | 32,077 rows/sec | 0.8 min elapsed
1,750,000 rows processed | 32,133 rows/sec | 0.9 min elapsed
2,000,000 rows processed | 32,191 rows/sec | 1.0 min elapsed
2,250,000 rows processed | 32,255 rows/sec | 1.2 min elapsed
2,500,000 rows processed | 32,308 rows/sec | 1.3 min elapsed
2,750,000 rows processed | 32,351 rows/sec | 1.4 min elapsed
3,000,000 rows processed | 32,380 rows/sec | 1.5 min elapsed
3,250,000 rows processed | 32,370 rows/sec | 1.7 min elapsed
3,500,000 rows processed | 32,382 rows/sec | 1.8 min elapsed
3,750,000 rows processed | 32,375 rows/sec | 1.9 min elapsed
4,000,000 rows processed | 32,346 rows/sec | 2.1 min elapsed
4,250,000 rows processed | 32,

In [10]:
# Preview of the decoded file
decoded_preview = pd.read_csv(DECODED_FILE, nrows=5, dtype=str, keep_default_na=False)
print(f"Decoded columns: {decoded_preview.shape[1]}")
decoded_preview

Decoded columns: 79


,YEAR,YEAR_LABEL,SAMPLE,SAMPLE_LABEL,SERIAL,CBSERIAL,HHWT,HHTYPE,HHTYPE_LABEL,CLUSTER,REGION,REGION_LABEL,STATEFIP,STATEFIP_LABEL,METRO,METRO_LABEL,STRATA,GQ,GQ_LABEL,OWNERSHP,OWNERSHP_LABEL,OWNERSHPD,OWNERSHPD_LABEL,RENT,RENT_LABEL,HHINCOME,VALUEH,VALUEH_LABEL,COUPLETYPE,COUPLETYPE_LABEL,PERNUM,PERWT,FAMSIZE,FAMSIZE_LABEL,NCHILD,NCHILD_LABEL,SEX,SEX_LABEL,AGE,AGE_LABEL,MARST,MARST_LABEL,MARRNO,MARRNO_LABEL,YRMARR,YRMARR_LABEL,DIVINYR,DIVINYR_LABEL,RACE,RACE_LABEL,RACED,RACED_LABEL,EDUC,EDUC_LABEL,EDUCD,EDUCD_LABEL,EMPSTAT,EMPSTAT_LABEL,EMPSTATD,EMPSTATD_LABEL,LABFORCE,LABFORCE_LABEL,CLASSWKR,CLASSWKR_LABEL,CLASSWKRD,CLASSWKRD_LABEL,WKSWORK2,WKSWORK2_LABEL,UHRSWORK,UHRSWORK_LABEL,INCTOT,FTOTINC,INCWAGE,INCEARN,POVERTY,MIGRATE1,MIGRATE1_LABEL,MIGRATE1D,MIGRATE1D_LABEL
0,2008,2008,200801,2008 ACS,1,23,57.0,1,Married-couple family household,2008000000011,32,East South Central Div.,01,Alabama,4,In metropolitan area: Central/principal city s...,130001,1,Households under 1970 definition,1,Owned or being bought (loan),13,Owned with mortgage or loan,0000,N/A,58100,0210000,,,,1,55.0,04,4,2,2,1,Male,037,37,1,"Married, spouse present",1,Married once,1998,,1,Blank (No),6,Other Asian or Pacific Islander,620,Korean,11,5+ years of college,114,Master's degree,1,Employed,10,At work,2,"Yes, in the labor force",2,Works for wages,27,State govt employee,6,50-52 weeks,40,40,58100,58100,57000,57000,271,1,Same house,10,Same house
1,2008,2008,200801,2008 ACS,1,23,57.0,1,Married-couple family household,2008000000011,32,East South Central Div.,01,Alabama,4,In metropolitan area: Central/principal city s...,130001,1,Households under 1970 definition,1,Owned or being bought (loan),13,Owned with mortgage or loan,0000,N/A,58100,0210000,,,,2,49.0,04,4,2,2,2,Female,036,36,1,"Married, spouse present",1,Married once,1998,,1,Blank (No),6,Other Asian or Pacific Islander,620,Korean,10,4 years of college,101,Bachelor's degree,3,Not in labor force,30,Not in Labor Force,1,"No, not in the labor force",0,N/A,00,N/A,0,N/A,00,N/A,0,58100,0,0,271,1,Same house,10,Same house
2,2008,2008,200801,2008 ACS,2,39,113.0,1,Married-couple family household,2008000000021,32,East South Central Div.,01,Alabama,4,In metropolitan area: Central/principal city s...,90101,1,Households under 1970 definition,1,Owned or being bought (loan),13,Owned with mortgage or loan,0000,N/A,125000,0230000,,,,1,109.0,02,2 family members present,0,0 children present,1,Male,030,30,1,"Married, spouse present",1,Married once,2006,,1,Blank (No),1,White,100,White,11,5+ years of college,114,Master's degree,1,Employed,10,At work,2,"Yes, in the labor force",2,Works for wages,22,"Wage/salary, private",6,50-52 weeks,40,40,75000,125000,75000,75000,501,1,Same house,10,Same house
3,2008,2008,200801,2008 ACS,2,39,113.0,1,Married-couple family household,2008000000021,32,East South Central Div.,01,Alabama,4,In metropolitan area: Central/principal city s...,90101,1,Households under 1970 definition,1,Owned or being bought (loan),13,Owned with mortgage or loan,0000,N/A,125000,0230000,,,,2,94.0,02,2 family members present,0,0 children present,2,Female,030,30,1,"Married, spouse present",1,Married once,2006,,1,Blank (No),1,White,100,White,10,4 years of college,101,Bachelor's degree,1,Employed,10,At work,2,"Yes, in the labor force",2,Works for wages,22,"Wage/salary, private",6,50-52 weeks,40,40,50000,125000,50000,50000,501,1,Same house,10,Same house
4,2008,2008,200801,2008 ACS,3,70,101.0,3,"Female householder, no husband present",2008000000031,32,East South Central Div.,01,Alabama,4,In metropolitan area: Central/principal city s...,90401,1,Households under 1970 definition,1,Owned or being bought (loan),12,Owned free and clear,0000,N/A,50000,0113000,,,,1,101.0,03,3,2,2,2,Female,040,40,3,Separated,1,Married once,2001,,1,Blank (No),2,Black/African American,200,Black/African American,08,2 years of college,081,"Associate's degree, type not specified",1,Employed,10,At work,2,"Yes, in the labor force",2,Works for wages,22,"Wage/salary, pri

## 3. Clean features

For every **categorical** feature the decoder produced two columns: the numeric IPUMS code and its readable label. Only the numeric duplicate is removed, e.g. `DIVINYR` is dropped and `DIVINYR_LABEL` is kept. All other columns are left unchanged.

Variables that are real numbers but also carry IPUMS labels (`YEAR`, `AGE`, `FAMSIZE`, `NCHILD`, `UHRSWORK`, `YRMARR`, `RENT`, `VALUEH`, `POVERTY`) are not categorical, so their numeric column is kept.

In [7]:
CLEAN_CHUNK_SIZE = 500_000

print(f"Loading: {DECODED_FILE}")
start = time.time()

all_columns = pd.read_csv(DECODED_FILE, nrows=0).columns.tolist()

# Drop every decoded label column.
# Keep the original IPUMS-coded variables.
drop_columns = [
    col for col in all_columns
    if col.endswith("_LABEL")
]

keep_columns = [
    col for col in all_columns
    if col not in drop_columns
]

print(f"Dropping {len(drop_columns)} _LABEL columns...")

# File is several GB, so read only the kept columns and write in chunks.
# dtype=str preserves original formatting, including leading zeros.
total_rows = 0

reader = pd.read_csv(
    DECODED_FILE,
    usecols=keep_columns,
    dtype=str,
    keep_default_na=False,
    chunksize=CLEAN_CHUNK_SIZE,
)

for i, chunk in enumerate(reader):
    # Preserve the original CSV column order.
    chunk = chunk[keep_columns]

    chunk.to_csv(
        CLEANED_FILE,
        index=False,
        mode="w" if i == 0 else "a",
        header=(i == 0),
    )

    total_rows += len(chunk)
    print(f"  processed {total_rows:,} rows...")

elapsed = time.time() - start

print()
print("========================================")
print("Cleaning complete")
print("========================================")
print(f"Rows:              {total_rows:,}")
print(f"Columns removed:   {len(drop_columns):,}")
print(f"Columns remaining: {len(keep_columns):,}")
print(f"Runtime:           {elapsed:.1f} seconds")

if CLEANED_FILE.exists():
    size_mb = CLEANED_FILE.stat().st_size / (1024 ** 2)
    print(f"Output size:       {size_mb:.1f} MB")

print()
print("Removed columns:")
for col in drop_columns:
    print(f"  - {col}")

print()
print("Remaining columns:")
for col in keep_columns:
    print(f"  - {col}")

Loading: c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008_decoded.csv
Dropping 33 _LABEL columns...
  processed 500,000 rows...
  processed 1,000,000 rows...
  processed 1,500,000 rows...
  processed 2,000,000 rows...
  processed 2,500,000 rows...
  processed 3,000,000 rows...
  processed 3,500,000 rows...
  processed 4,000,000 rows...
  processed 4,500,000 rows...
  processed 5,000,000 rows...
  processed 5,500,000 rows...
  processed 5,641,835 rows...

Cleaning complete
Rows:              5,641,835
Columns removed:   33
Columns remaining: 46
Runtime:           74.3 seconds
Output size:       982.9 MB

Removed columns:
  - YEAR_LABEL
  - SAMPLE_LABEL
  - HHTYPE_LABEL
  - REGION_LABEL
  - STATEFIP_LABEL
  - METRO_LABEL
  - GQ_LABEL
  - OWNERSHP_LABEL
  - OWNERSHPD_LABEL
  - RENT_LABEL
  - VALUEH_LABEL
  - COUPLETYPE_LABEL
  - FAMSIZE_LABEL
  - NCHILD_LABEL
  - SEX_LABEL
  - AGE_LABEL
  - MARST_LABEL
  - MARRNO_LABEL
  - YRMARR_LABEL
  - DIVINYR_LABEL
  - RACE_LABEL
  - RACED

In [8]:
# Preview of the cleaned dataset
cleaned_preview = pd.read_csv(CLEANED_FILE, nrows=10)
print(f"Cleaned columns: {cleaned_preview.shape[1]}")
cleaned_preview

Cleaned columns: 46


,YEAR,SAMPLE,SERIAL,CBSERIAL,HHWT,HHTYPE,CLUSTER,REGION,STATEFIP,METRO,STRATA,GQ,OWNERSHP,OWNERSHPD,RENT,HHINCOME,VALUEH,COUPLETYPE,PERNUM,PERWT,FAMSIZE,NCHILD,SEX,AGE,MARST,MARRNO,YRMARR,DIVINYR,RACE,RACED,EDUC,EDUCD,EMPSTAT,EMPSTATD,LABFORCE,CLASSWKR,CLASSWKRD,WKSWORK2,UHRSWORK,INCTOT,FTOTINC,INCWAGE,INCEARN,POVERTY,MIGRATE1,MIGRATE1D
0,2008,200801,1,23,57.0,1,2008000000011,32,1,4,130001,1,1,13,0,58100,210000,NaN,1,55.0,4,2,1,37,1,1,1998,1,6,620,11,114,1,10,2,2,27,6,40,58100,58100,57000,57000,271,1,10
1,2008,200801,1,23,57.0,1,2008000000011,32,1,4,130001,1,1,13,0,58100,210000,NaN,2,49.0,4,2,2,36,1,1,1998,1,6,620,10,101,3,30,1,0,0,0,0,0,58100,0,0,271,1,10
2,2008,200801,2,39,113.0,1,2008000000021,32,1,4,90101,1,1,13,0,125000,230000,NaN,1,109.0,2,0,1,30,1,1,2006,1,1,100,11,114,1,10,2,2,22,6,40,75000,125000,75000,75000,501,1,10
3,2008,200801,2,39,113.0,1,2008000000021,32,1,4,90101,1,1,13,0,125000,230000,NaN,2,94.0,2,0,2,30,1,1,2006,1,1,100,10,101,1,10,2,2,22,6,40,50000,125000,50000,50000,501,1,10
4,2008,200801,3,70,101.0,3,2008000000031,32,1,4,90401,1,1,12,0,50000,113000,NaN,1,101.0,3,2,2,40,3,1,2001,1,2,200,8,81,1,10,2,2,22,6,40,50000,50000,50000,50000,294,1,10
5,2008,200801,4,122,83.0,1,2008000000041,32,1,1,110001,1,1,13,0,45480,120000,NaN,1,83.0,3,1,1,55,1,2,1974,1,1,100,8,81,3,30,1,0,0,0,0,19980,45480,0,0,267,1,10
6,2008,200801,4,122,83.0,1,2008000000041,32,1,1,110001,1,1,13,0,45480,120000,NaN,2,83.0,3,1,2,50,1,2,1974,1,1,100,6,63,1,10,2,2,22,6,40,25500,45480,25500,25500,267,1,10
7,2008,200801,5,204,117.0,6,2008000000051,32,1,3,230001,1,1,13,0,27000,215000,NaN,1,115.0,1,0,2,22,4,1,2005,2,1,100,6,65,1,10,2,1,13,5,40,27000,27000,0,27000,245,2,23
8,2008,200801,8,423,21.0,6,2008000000081,32,1,1,40001,1,2,22,70,7600,9999999,NaN,1,21.0,1,0,2,59,4,3,1977,1,1,100,0,2,3,30,1,0,0,0,0,7600,7600,0,0,69,1,10
9,2008,200801,9,429,75.0,1,2008000000091,32,1,3,200001,1,1,13,0,81000,180000,NaN,1,76.0,3,1,1,46,1,1,1984,1,1,100,6,65,1,10,2,2,22,6,40,50000,81000,50000,50000,490,1,10


## 4. Drop unused features

Remove technical / survey-design fields and detailed versions of variables that are not used in the analysis:

* **Technical fields:** `CBSERIAL`, `PERNUM`, `CLUSTER`, `STRATA`, `GQ`
* **Detailed versions** (the general version is kept): `RACED`, `EDUCD`, `OWNERSHPD`, `EMPSTATD`, `CLASSWKRD`, `MIGRATE1D`

The cleaned file is rewritten in place.

In [17]:
unused_features = [

    # -----------------------------------------
    # Technical fields not used as predictors
    # -----------------------------------------
    "CBSERIAL",
    "PERNUM",
    "CLUSTER",
    "STRATA",
    "GQ",
    "SAMPLE",

    # -----------------------------------------
    # Detailed versions of general variables
    # -----------------------------------------
    "RACED",
    "EDUCD",
    "OWNERSHPD",
    "EMPSTATD",
    "CLASSWKRD",
    "MIGRATE1D",
]

print(f"Loading: {CLEANED_FILE}")
start = time.time()

all_columns = pd.read_csv(CLEANED_FILE, nrows=0).columns.tolist()

drop_columns = [
    col for col in unused_features
    if col in all_columns
]

keep_columns = [
    col for col in all_columns
    if col not in drop_columns
]

print(f"Dropping {len(drop_columns)} unused features...")

# Write to a temporary file, then replace the cleaned file.
temp_file = CLEANED_FILE.with_suffix(".tmp.csv")
total_rows = 0

reader = pd.read_csv(
    CLEANED_FILE,
    usecols=keep_columns,
    dtype=str,
    keep_default_na=False,
    chunksize=CLEAN_CHUNK_SIZE,
)

for i, chunk in enumerate(reader):
    # Preserve the original CSV column order.
    chunk = chunk[keep_columns]

    chunk.to_csv(
        temp_file,
        index=False,
        mode="w" if i == 0 else "a",
        header=(i == 0),
    )

    total_rows += len(chunk)
    print(f"  processed {total_rows:,} rows...")

temp_file.replace(CLEANED_FILE)

elapsed = time.time() - start

print()
print("========================================")
print("Unused features removed")
print("========================================")
print(f"Rows:              {total_rows:,}")
print(f"Columns removed:   {len(drop_columns):,}")
print(f"Columns remaining: {len(keep_columns):,}")
print(f"Runtime:           {elapsed:.1f} seconds")

size_mb = CLEANED_FILE.stat().st_size / (1024 ** 2)
print(f"Output size:       {size_mb:.1f} MB")

print()
print("Removed columns:")
for col in drop_columns:
    print(f"  - {col}")

print()
print("Remaining columns:")
for col in keep_columns:
    print(f"  - {col}")

Loading: c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008_cleaned.csv
Dropping 1 unused features...
  processed 500,000 rows...
  processed 1,000,000 rows...
  processed 1,500,000 rows...
  processed 2,000,000 rows...
  processed 2,500,000 rows...
  processed 3,000,000 rows...
  processed 3,500,000 rows...
  processed 4,000,000 rows...
  processed 4,500,000 rows...
  processed 5,000,000 rows...
  processed 5,500,000 rows...
  processed 6,000,000 rows...
  processed 6,500,000 rows...
  processed 7,000,000 rows...
  processed 7,500,000 rows...
  processed 7,630,587 rows...

Unused features removed
Rows:              7,630,587
Columns removed:   1
Columns remaining: 34
Runtime:           63.4 seconds
Output size:       871.9 MB

Removed columns:
  - SAMPLE

Remaining columns:
  - YEAR
  - SERIAL
  - HHWT
  - HHTYPE
  - REGION
  - STATEFIP
  - METRO
  - OWNERSHP
  - RENT
  - HHINCOME
  - VALUEH
  - COUPLETYPE
  - PERWT
  - FAMSIZE
  - NCHILD
  - SEX
  - AGE
  - MARST
  - MARRNO


In [18]:
# Preview of the final dataset
final_preview = pd.read_csv(CLEANED_FILE, nrows=10)
print(f"Final columns: {final_preview.shape[1]}")
final_preview

Final columns: 34


,YEAR,SERIAL,HHWT,HHTYPE,REGION,STATEFIP,METRO,OWNERSHP,RENT,HHINCOME,VALUEH,COUPLETYPE,PERWT,FAMSIZE,NCHILD,SEX,AGE,MARST,MARRNO,YRMARR,DIVINYR,RACE,EDUC,EMPSTAT,LABFORCE,CLASSWKR,WKSWORK2,UHRSWORK,INCTOT,FTOTINC,INCWAGE,INCEARN,POVERTY,MIGRATE1
0,2008,1,57.0,1,32,1,4,1,0,58100,210000,NaN,55.0,4,2,1,37,1,1,1998,1,6,11,1,2,2,6,40,58100,58100,57000,57000,271,1
1,2008,1,57.0,1,32,1,4,1,0,58100,210000,NaN,49.0,4,2,2,36,1,1,1998,1,6,10,3,1,0,0,0,0,58100,0,0,271,1
2,2008,2,113.0,1,32,1,4,1,0,125000,230000,NaN,109.0,2,0,1,30,1,1,2006,1,1,11,1,2,2,6,40,75000,125000,75000,75000,501,1
3,2008,2,113.0,1,32,1,4,1,0,125000,230000,NaN,94.0,2,0,2,30,1,1,2006,1,1,10,1,2,2,6,40,50000,125000,50000,50000,501,1
4,2008,3,101.0,3,32,1,4,1,0,50000,113000,NaN,101.0,3,2,2,40,3,1,2001,1,2,8,1,2,2,6,40,50000,50000,50000,50000,294,1
5,2008,4,83.0,1,32,1,1,1,0,45480,120000,NaN,83.0,3,1,1,55,1,2,1974,1,1,8,3,1,0,0,0,19980,45480,0,0,267,1
6,2008,4,83.0,1,32,1,1,1,0,45480,120000,NaN,83.0,3,1,2,50,1,2,1974,1,1,6,1,2,2,6,40,25500,45480,25500,25500,267,1
7,2008,5,117.0,6,32,1,3,1,0,27000,215000,NaN,115.0,1,0,2,22,4,1,2005,2,1,6,1,2,1,5,40,27000,27000,0,27000,245,2
8,2008,8,21.0,6,32,1,1,2,70,7600,9999999,NaN,21.0,1,0,2,59,4,3,1977,1,1,0,3,1,0,0,0,7600,7600,0,0,69,1
9,2008,9,75.0,1,32,1,3,1,0,81000,180000,NaN,76.0,3,1,1,46,1,1,1984,1,1,6,1,2,2,6,40,50000,81000,50000,50000,490,1
